# GRACE – Guaranteed false-dismissal control for time-adaptive transient stability assessment
Google Colab version of the full pipeline used in the manuscript:

1. simulate IEEE 39-bus fault contingencies with **ANDES**,
2. train the predictors (DD, MT, PI, PI-0, gradient boosting),
3. run class-conditional **conformal risk control** with 200 calibration re-splits,
4. reproduce all figures and the key numbers of Tables 2–4.

**Runtime (CPU runtime is enough, no GPU needed):** with `N_SIMS = 1200` the simulations take about 20–40 min (depending on the number of CPU cores Colab provides), training about 15 min and the analysis about 10 min. Set `QUICK_TEST = True` for a short smoke test with 200 contingencies (the numbers will then differ from the paper).

Colab sessions can disconnect. With `USE_DRIVE = True` all files are kept in your Google Drive, so you can resume: contingencies and trained experiments that already exist are skipped automatically.

In [ ]:
#@title 1. Settings
USE_DRIVE = True        #@param {type:"boolean"}
QUICK_TEST = False      #@param {type:"boolean"}
N_SIMS = 1200           #@param {type:"integer"}
ITERS = 700             #@param {type:"integer"}
if QUICK_TEST:
    N_SIMS, ITERS = 200, 300
print("N_SIMS =", N_SIMS, " ITERS =", ITERS)

In [ ]:
#@title 2. Working directory (Google Drive optional)
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORKDIR = '/content/drive/MyDrive/GRACE_TSA'
else:
    WORKDIR = '/content/GRACE_TSA'
for d in ('', 'figs', 'paper'):
    os.makedirs(os.path.join(WORKDIR, d), exist_ok=True)
os.chdir(WORKDIR)
print("Working in", os.getcwd())

In [ ]:
#@title 3. Install dependencies (about 1–2 min)
!pip -q install "andes==1.8.*" autograd
!apt-get -qq install -y fonts-liberation > /dev/null
import matplotlib.font_manager as fm; fm._load_fontmanager(try_read_cache=False)
import andes, autograd, sklearn
print("ANDES", andes.__version__, "| scikit-learn", sklearn.__version__, "| CPU cores:", os.cpu_count())

## 4. Source code
Each cell below writes one module of the pipeline to the working directory. Run them all once.

In [ ]:
%%writefile gen.py
"""Scenario generation: IEEE 39-bus, three-phase line faults cleared by line tripping (ANDES)."""
import andes, numpy as np, sys, time, os, pickle
andes.config_logger(stream_level=50)
CASE = 'ieee39.json'
T_FAULT, DT, POST = 0.1, 0.01, 3.0

def eligible_lines():
    ss = andes.load(CASE, setup=False, no_output=True, default_config=True)
    L = ss.Line.as_df()
    lines = []
    edges = [(int(a), int(b)) for a, b in zip(L.bus1, L.bus2)]
    buses = set(int(b) for b in ss.Bus.idx.v)
    for k, row in L.iterrows():
        if row.trans == 1: continue
        rest = [e for j, e in enumerate(edges) if j != k]
        adj = {b: set() for b in buses}
        for a, b in rest: adj[a].add(b); adj[b].add(a)
        seen, st = {1}, [1]
        while st:
            u = st.pop()
            for v in adj[u]:
                if v not in seen: seen.add(v); st.append(v)
        if len(seen) == len(buses): lines.append((row.idx, int(row.bus1), int(row.bus2)))
    return lines

def simulate(sc):
    ss = andes.load(CASE, setup=False, no_output=True, default_config=True)
    rs = np.random.RandomState(sc["seed"])
    lam = sc["lam"]
    tc = T_FAULT + sc["tau"]
    ss.add('Fault', dict(bus=sc["fbus"], tf=T_FAULT, tc=tc, xf=1e-4, rf=0.0))
    ss.add('Toggle', dict(model='Line', dev=sc["line"], t=tc))
    ss.setup()
    ss.PQ.p0.v[:] = ss.PQ.p0.v * lam * (1 + rs.uniform(-0.1, 0.1, ss.PQ.n))
    ss.PQ.q0.v[:] = ss.PQ.q0.v * lam * (1 + rs.uniform(-0.1, 0.1, ss.PQ.n))
    ss.PV.p0.v[:] = ss.PV.p0.v * lam * (1 + rs.uniform(-0.1, 0.1, ss.PV.n))
    ss.PFlow.run()
    if not ss.PFlow.converged: return None
    ss.TDS.config.tf = tc + POST; ss.TDS.config.tstep = DT; ss.TDS.config.no_tqdm = 1
    ss.TDS.config.criteria = 1; ss.TDS.config.ddelta_limit = 540
    ss.TDS.run()
    t = np.asarray(ss.dae.ts.t)
    g = ss.GENROU
    X = np.asarray(ss.dae.ts.x); Y = np.asarray(ss.dae.ts.y)
    d, w = X[:, g.delta.a], X[:, g.omega.a]
    te, tm = Y[:, g.te.a], Y[:, g.tm.a]
    grid = np.arange(0, tc + POST + 1e-9, DT)
    grid = grid[grid <= t[-1] + 1e-9]
    # remove duplicate event instants (keep post-event value), then linear interpolation onto the grid
    keep = np.r_[np.diff(t) > 1e-9, True]
    t, X, Y = t[keep], X[keep], Y[keep]
    d, w = X[:, g.delta.a], X[:, g.omega.a]
    te, tm = Y[:, g.te.a], Y[:, g.tm.a]
    def rs_(A):
        return np.stack([np.interp(grid, t, A[:, j]) for j in range(A.shape[1])], 1).astype(np.float32)
    D, W, TE, TM = rs_(d), rs_(w), rs_(te), rs_(tm)
    M = np.asarray(g.M.v, dtype=float)
    dcoi = (D * M).sum(1, keepdims=True) / M.sum()
    sep = np.degrees(np.max(D, 1) - np.min(D, 1))
    dmax = float(sep[grid >= tc - 1e-9].max()) if (grid >= tc - 1e-9).any() else float(sep.max())
    stopped = t[-1] < tc + POST - 1e-6
    unstable = (dmax >= 360.0) or stopped
    tsi = (360 - dmax) / (360 + dmax)
    return dict(sc=sc, tc=tc, t=grid.astype(np.float32), delta=D, omega=W, te=TE, tm=TM,
                dmax=dmax, tsi=float(tsi), unstable=bool(unstable), stopped=bool(stopped),
                converged=bool(ss.TDS.converged), M=M, Sn=np.asarray(g.Sn.v, float))

def scenarios(n_total):
    lines = eligible_lines()
    rng = np.random.RandomState(2026)
    scen = []
    for i in range(n_total):
        ln = lines[rng.randint(len(lines))]
        scen.append(dict(id=i, seed=10000 + i, lam=float(rng.uniform(0.80, 1.20)), line=ln[0],
                         fbus=ln[1 + rng.randint(2)], tau=float(rng.uniform(0.05, 0.35))))
    return scen

def run_one(sc):
    fn = f"sims/s{sc['id']:05d}.pkl"
    if os.path.exists(fn): return sc["id"]
    try:
        r = simulate(sc)
    except Exception as e:
        r = dict(sc=sc, error=str(e))
    tmp = fn + ".tmp"
    pickle.dump(r, open(tmp, "wb")); os.replace(tmp, fn)
    return sc["id"]

if __name__ == "__main__":
    from concurrent.futures import ProcessPoolExecutor
    n_total = int(sys.argv[1]); n_jobs = int(sys.argv[2]) if len(sys.argv) > 2 else os.cpu_count()
    os.makedirs("sims", exist_ok=True)
    todo = [s for s in scenarios(n_total) if not os.path.exists(f"sims/s{s['id']:05d}.pkl")]
    print(f"{len(todo)} contingencies to simulate with {n_jobs} processes", flush=True)
    t0 = time.time(); done = 0
    with ProcessPoolExecutor(n_jobs) as ex:
        for _ in ex.map(run_one, todo, chunksize=4):
            done += 1
            if done % 50 == 0 or done == len(todo):
                print(f"{done}/{len(todo)}  elapsed {time.time() - t0:.0f} s", flush=True)


In [ ]:
%%writefile data.py
"""Build windowed datasets from simulation records."""
import numpy as np, pickle, glob
DT, WB = 0.01, 2 * np.pi * 60
WINDOWS = [3, 6, 10, 15]          # samples after clearing: 30, 60, 100, 150 ms
H = 25                            # forecast horizon (samples) = 0.25 s

def load_all(path="sims"):
    recs = []
    for f in sorted(glob.glob(f"{path}/s*.pkl")):
        try:
            r = pickle.load(open(f, "rb"))
        except Exception:
            continue
        if r is None or "error" in r: continue
        recs.append(r)
    return recs

def build(recs, W, noise=None, rng=None):
    """Return inputs X (N, W, 10, 3), Pm0 (N,10), targets: tsi, y (N,H,10,3), mask (N,H), meta."""
    Xs, P, T, Y, Mk, meta = [], [], [], [], [], []
    for r in recs:
        M = r["M"]; Mt = M.sum()
        d = r["delta"].astype(float); w = r["omega"].astype(float) - 1.0; te = r["te"].astype(float)
        pm0 = te[5]                                    # pre-fault electrical power = Pm estimate
        dc = d - (d * M).sum(1, keepdims=True) / Mt    # COI-referenced angle
        i0 = int(np.ceil(r["tc"] / DT - 1e-6))
        n = len(r["t"])
        if i0 + W > n: continue
        x = np.stack([dc, 100 * w, te - pm0], -1)       # (n,10,3)
        win = x[i0:i0 + W].copy()
        if noise is not None:
            win[..., 0] += rng.normal(0, noise[0], win[..., 0].shape)
            win[..., 1] += rng.normal(0, noise[1], win[..., 1].shape)
            win[..., 2] += rng.normal(0, noise[2], win[..., 2].shape)
        y = np.zeros((H + 1, 10, 3)); m = np.zeros(H + 1)
        seg = x[i0 + W - 1:i0 + W + H]; y[:len(seg)] = seg; m[:len(seg)] = 1
        Xs.append(win); P.append(pm0); T.append(r["tsi"]); Y.append(y); Mk.append(m)
        meta.append((r["sc"]["id"], r["sc"]["line"], r["sc"]["lam"], r["sc"]["tau"], r["unstable"], r["sc"]["fbus"]))
    meta = np.array(meta, dtype=object)
    return dict(X=np.array(Xs), pm0=np.array(P), tsi=np.array(T), Y=np.array(Y), mask=np.array(Mk),
                id=meta[:, 0].astype(int), line=meta[:, 1].astype(str), lam=meta[:, 2].astype(float),
                tau=meta[:, 3].astype(float), unstable=meta[:, 4].astype(bool), fbus=meta[:, 5].astype(int))


In [ ]:
%%writefile model.py
"""Swing-residual multi-task network (SwingNet) implemented with autograd."""
import autograd.numpy as np
from autograd import grad
from autograd.misc import flatten
import numpy as onp
DT, WB = 0.01, 2 * onp.pi * 60

def init(sizes, rs):
    return [(rs.randn(m, n) * onp.sqrt(1.0 / m), onp.zeros(n)) for m, n in zip(sizes[:-1], sizes[1:])]

def mlp(p, h):
    for W, b in p[:-1]: h = np.tanh(h @ W + b)
    W, b = p[-1]; return h @ W + b

class SwingNet:
    def __init__(self, W, H, M, K=None, hidden=128, beta_f=1.0, beta_p=1.0, seed=0, heads=True, hard=False, lam_traj=1.0):
        self.W, self.H, self.M = W, H, onp.asarray(M)
        self.K = onp.zeros_like(self.M) if K is None else onp.asarray(K)
        self.bf, self.bp, self.heads, self.hard, self.lt = beta_f, beta_p, heads, hard, lam_traj
        rs = onp.random.RandomState(seed)
        din = W * 30 + 10
        self.p = dict(trunk=init([din, hidden, hidden], rs), m=init([hidden, 1], rs),
                      f=init([hidden, H * (10 if hard else 30)], rs))
        for k in ("m", "f"):
            self.p[k][-1] = (self.p[k][-1][0] * 0.1, self.p[k][-1][1])

    def _inputs(self, X, pm0):
        Z = (X - self.mu) / self.sd
        return onp.concatenate([Z.reshape(len(X), -1), (pm0 - self.pmu) / self.psd], 1)

    def fit_scalers(self, X, pm0, Y, mask):
        self.mu = X.reshape(-1, 10, 3).mean(0); self.sd = X.reshape(-1, 10, 3).std(0) + 1e-6
        self.pmu, self.psd = pm0.mean(0), pm0.std(0) + 1e-6
        dY = onp.diff(Y, axis=1)                          # step-wise increments, per channel scale
        self.ysd = onp.array([dY[..., c].std() for c in range(3)]) * onp.sqrt(self.H) + 1e-6
        # residual scales from the true trajectories (measures model mismatch as well)
        rA, rB = self.residual_np(Y, pm0)
        m = mask[:, 1:, None] * mask[:, :-1, None]
        self.rAs = onp.sqrt((rA ** 2 * m).sum() / (m.sum() * 10)) + 1e-9
        self.rBs = onp.sqrt((rB ** 2 * m).sum() / (m.sum() * 10)) + 1e-9
        self.true_res = (self.rAs, self.rBs)

    def forward(self, p, Xin, last):
        h = mlp(p["trunk"], Xin)
        h = np.tanh(h)
        m = mlp(p["m"], h)[:, 0]
        if not self.hard:
            inc = mlp(p["f"], h).reshape(len(Xin), self.H, 10, 3) * (self.ysd / onp.sqrt(self.H))
            traj = np.concatenate([last[:, None], last[:, None] + np.cumsum(inc, axis=1)], 1)
            return m, traj
        # physics-structured head: predict speed increments; angle and power follow from the swing equation
        inc = mlp(p["f"], h).reshape(len(Xin), self.H, 10) * (self.ysd[1] / onp.sqrt(self.H))
        w = np.concatenate([last[:, None, :, 1], last[:, None, :, 1] + np.cumsum(inc, axis=1)], 1) / 100.0
        wc = w - np.sum(w * self.M, -1, keepdims=True) / self.M.sum()
        dd = DT * WB * 0.5 * (wc[:, 1:] + wc[:, :-1])
        d = np.concatenate([last[:, None, :, 0], last[:, None, :, 0] + np.cumsum(dd, axis=1)], 1)
        pmid = -self.M * (w[:, 1:] - w[:, :-1]) / DT - self.K * 0.5 * (w[:, 1:] + w[:, :-1])
        p_full = np.concatenate([last[:, None, :, 2], pmid], 1)       # index h>=1 holds midpoint power of step h-1->h
        traj = np.stack([d, 100.0 * w, p_full], -1)
        return m, traj

    def residual(self, traj, pm0):
        d, w, dp = traj[..., 0], traj[..., 1] / 100.0, traj[..., 2]
        wcoi = np.sum(w * self.M, -1, keepdims=True) / self.M.sum()
        rA = (d[:, 1:] - d[:, :-1]) / DT - WB * 0.5 * ((w - wcoi)[:, 1:] + (w - wcoi)[:, :-1])
        rB = self.M * (w[:, 1:] - w[:, :-1]) / DT + 0.5 * (dp[:, 1:] + dp[:, :-1]) \
             + self.K * 0.5 * (w[:, 1:] + w[:, :-1])
        return rA, rB

    def residual_np(self, Y, pm0):
        return self.residual(Y, pm0)

    def loss(self, flat, Xin, last, pm0, tsi, Y, mask, Xu=None, lastu=None, pmu=None):
        p = self.unflat(flat)
        m, traj = self.forward(p, Xin, last)
        e = m - tsi
        L = np.mean(np.where(np.abs(e) < 0.1, 0.5 * e ** 2, 0.1 * (np.abs(e) - 0.05))) / 0.01
        if self.bf > 0:
            Yt = Y
            if self.hard:
                Yt = np.concatenate([Y[..., :2], np.concatenate([Y[:, :1, :, 2:], 0.5 * (Y[:, 1:, :, 2:] + Y[:, :-1, :, 2:])], 1)], -1)
            err = ((traj[:, 1:] - Yt[:, 1:]) / self.ysd) ** 2
            L = L + self.lt * self.bf * np.sum(err * mask[:, 1:, None, None]) / (np.sum(mask[:, 1:]) * 30)
        if self.bp > 0 and not self.hard:
            rA, rB = self.residual(traj, pm0)
            Lp = np.mean((rA / self.rAs) ** 2) + np.mean((rB / self.rBs) ** 2)
            if Xu is not None:
                _, tu = self.forward(p, Xu, lastu)
                rA, rB = self.residual(tu, pmu)
                Lp = 0.5 * Lp + 0.5 * (np.mean((rA / self.rAs) ** 2) + np.mean((rB / self.rBs) ** 2))
            L = L + self.bp * 0.01 * Lp
        l2 = sum(np.sum(W ** 2) for W, _ in p["trunk"]) * 1e-5
        return L + l2

    def fit(self, D, iters=1500, lr=2e-3, U=None, batch=256, seed=0, verbose=False):
        self.fit_scalers(D["X"], D["pm0"], D["Y"], D["mask"])
        flat, self.unflat = flatten(self.p)
        Xin = self._inputs(D["X"], D["pm0"]); last = D["Y"][:, 0]
        if U is not None: Xu = self._inputs(U["X"], U["pm0"]); lastu = U["X"][:, -1]
        g = grad(self.loss)
        mm = onp.zeros_like(flat); vv = onp.zeros_like(flat)
        rs = onp.random.RandomState(seed); n = len(Xin)
        for i in range(1, iters + 1):
            b = rs.choice(n, min(batch, n), replace=False)
            args = [Xin[b], last[b], D["pm0"][b], D["tsi"][b], D["Y"][b], D["mask"][b]]
            if U is not None:
                bu = rs.choice(len(Xu), min(batch, len(Xu)), replace=False)
                args += [Xu[bu], lastu[bu], U["pm0"][bu]]
            gi = g(flat, *args)
            lr_i = lr * (0.5 * (1 + onp.cos(onp.pi * i / iters)) * 0.95 + 0.05)
            mm = 0.9 * mm + 0.1 * gi; vv = 0.999 * vv + 0.001 * gi ** 2
            flat = flat - lr_i * (mm / (1 - 0.9 ** i)) / (onp.sqrt(vv / (1 - 0.999 ** i)) + 1e-8)
            if verbose and i % 250 == 0: print(i, float(self.loss(flat, *args)))
        self.flat = flat
        return self

    def predict(self, D):
        p = self.unflat(self.flat)
        m, traj = self.forward(p, self._inputs(D["X"], D["pm0"]), D["X"][:, -1])
        return onp.asarray(m), onp.asarray(traj)


In [ ]:
%%writefile exp.py
"""Experiments: splits, model variants, fixed-window metrics, time-adaptive conformal risk control."""
import numpy as np, pickle, time, sys, json
from sklearn.ensemble import HistGradientBoostingClassifier
from data import load_all, build, WINDOWS, H
from model import SwingNet

R = load_all()
M = R[0]["M"]; Sn = R[0]["Sn"]
K_GOV = (1.0 / 2.1) * Sn / (0.05 * 100.0)       # short-term droop gain: (T2/T3) * Sn / (R * Sbase)
ids = np.array([r["sc"]["id"] for r in R]); lines = np.array([r["sc"]["line"] for r in R])
lam = np.array([r["sc"]["lam"] for r in R])
rng = np.random.RandomState(7)

def split_random():
    p = rng.permutation(len(R)); n = len(R)
    return dict(train=ids[p[:int(.6 * n)]], cal=ids[p[int(.6 * n):int(.8 * n)]], test=ids[p[int(.8 * n):]])

def split_lines():
    ul = np.array(sorted(set(lines))); rs = np.random.RandomState(11); p = rs.permutation(len(ul))
    tr_l, other_l = ul[p[:19]], ul[p[19:]]
    tr_ids = ids[np.isin(lines, tr_l)]; rs.shuffle(tr_ids)
    k = int(0.85 * len(tr_ids))
    return dict(train=tr_ids[:k], cal_naive=tr_ids[k:], heldout=ids[np.isin(lines, other_l)],
                train_lines=list(tr_l), heldout_lines=list(other_l))

def split_op():
    tr = ids[lam <= 1.04].copy(); rs = np.random.RandomState(13); rs.shuffle(tr); k = int(0.85 * len(tr))
    return dict(train=tr[:k], cal_naive=tr[k:], cal=ids[(lam > 1.04) & (lam <= 1.10)], test=ids[lam > 1.10])

def subset(D, sel):
    m = np.isin(D["id"], sel)
    return {k: (v[m] if isinstance(v, np.ndarray) and len(v) == len(D["id"]) else v) for k, v in D.items()}

def feats(D):
    return np.concatenate([D["X"].reshape(len(D["X"]), -1), D["pm0"]], 1)

def train_variant(kind, Dtr, W, seed, iters=800, U=None):
    if kind == "DD":   net = SwingNet(W, H, M, K_GOV, hidden=96, beta_f=0.0, beta_p=0.0, seed=seed)
    elif kind == "MT": net = SwingNet(W, H, M, K_GOV, hidden=96, beta_f=1.0, beta_p=0.0, seed=seed)
    elif kind == "PI": net = SwingNet(W, H, M, K_GOV, hidden=96, beta_f=1.0, beta_p=1.0, seed=seed)
    elif kind == "PI0": net = SwingNet(W, H, M, None, hidden=96, beta_f=1.0, beta_p=1.0, seed=seed)  # no governor term
    return net.fit(Dtr, iters=iters, batch=128, seed=seed, U=U)

def metrics(m, D):
    y = D["unstable"]; yhat = m < 0
    return dict(acc=float((yhat == y).mean()), fdr=float((~yhat & y).sum() / max(y.sum(), 1)),
                far=float((yhat & ~y).sum() / max((~y).sum(), 1)), mae=float(np.abs(m - D["tsi"]).mean()))

if __name__ == "__main__":
    which = sys.argv[1]
    out = {}
    DS = {W: build(R, W) for W in WINDOWS}
    t0 = time.time()
    if which == "S1":
        sp = split_random(); seeds = [0]; kinds = ["DD", "MT", "PI"]
    elif which == "S2":
        sp = split_lines(); seeds = [0, 1, 2]; kinds = ["DD", "MT", "PI"]
    elif which == "S3":
        sp = split_op(); seeds = [0]; kinds = ["DD", "MT", "PI"]
    elif which == "S2U":
        sp = split_lines(); seeds = [0]; kinds = ["PIU"]
    elif which == "S2G":
        sp = split_lines(); seeds = [0]; kinds = ["PI0"]
    elif which in ("S2L", "S2F"):
        sp = split_lines(); seeds = [0]
    iters = int(sys.argv[2]) if len(sys.argv) > 2 else 800
    preds = {}
    NOISE = (0.01, 0.01, 0.02)
    DSn = {W: build(R, W, noise=NOISE, rng=np.random.RandomState(99)) for W in WINDOWS} if which == "S2" else None
    if which in ("S2L", "S2F"):
        W = 10; D = DS[W]; Dtr = subset(D, sp["train"])
        if which == "S2L":
            for lp in (0.001, 0.1, 1.0):
                net = SwingNet(W, H, M, K_GOV, hidden=96, beta_f=1.0, beta_p=lp / 0.01, seed=0).fit(Dtr, iters=iters, batch=128, seed=0)
                preds[("PI_lp", lp)] = dict(m=net.predict(D)[0], id=D["id"]); print("lp", lp, flush=True)
        else:
            rs = np.random.RandomState(3); tr = sp["train"].copy(); rs.shuffle(tr)
            for frac in (0.25, 0.5):
                Dsub = subset(D, tr[: int(frac * len(tr))])
                for kind in ("DD", "MT", "PI"):
                    net = train_variant(kind, Dsub, W, 0, iters=iters)
                    preds[(kind, frac)] = dict(m=net.predict(D)[0], id=D["id"]); print(kind, frac, flush=True)
        pickle.dump(dict(split=sp, preds=preds), open(f"res_{which}.pkl", "wb")); sys.exit()
    for W in WINDOWS:
        D = DS[W]; Dtr = subset(D, sp["train"])
        for kind in kinds:
            for s in seeds:
                ts = time.time()
                if kind == "PIU":
                    rs = np.random.RandomState(5); ho = sp["heldout"].copy(); rs.shuffle(ho)
                    pool = ho[: len(ho) // 2]
                    net = SwingNet(W, H, M, K_GOV, hidden=96, beta_f=1.0, beta_p=1.0, seed=s)
                    net.fit(Dtr, iters=iters, batch=128, seed=s, U=subset(D, pool))
                    preds[("pool_ids",)] = pool
                else:
                    net = train_variant(kind, Dtr, W, s, iters=iters)
                ttrain = time.time() - ts
                m, traj = net.predict(D)
                ti = time.time(); net.predict(subset(D, D["id"][:1])); tinf = time.time() - ti
                mres = net.residual(traj[:, :], D["pm0"]) if kind != "DD" else None
                phys = None if mres is None else (float(np.sqrt(np.mean(mres[0] ** 2))), float(np.sqrt(np.mean(mres[1] ** 2))))
                mtr = net.predict(Dtr)[0]; scale = float(np.median(np.abs(mtr - Dtr["tsi"])))
                preds[(kind, W, s)] = dict(m=m, id=D["id"], ttrain=ttrain, tinf=tinf, scale=scale, phys=phys,
                                            traj=traj.astype(np.float32) if (W == 10 and s == 0) else None,
                                            m_noise=(net.predict(DSn[W])[0] if (DSn is not None and s == 0) else None),
                                            true_res=getattr(net, "true_res", None))
                print(which, kind, W, s, round(ttrain, 1), flush=True)
        # gradient-boosting classifier baselines (also biased pair for time-adaptive consistency baseline)
        Xtr = feats(Dtr); ytr = Dtr["unstable"].astype(int)
        for name, cw in (("GB", None), ("GBu", {0: 1, 1: 5}), ("GBs", {0: 5, 1: 1})):
            clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.08, class_weight=cw, random_state=0)
            ts = time.time(); clf.fit(Xtr, ytr); ttrain = time.time() - ts
            pr = clf.predict_proba(feats(D))[:, 1]
            preds[(name, W, 0)] = dict(p=pr, id=D["id"], ttrain=ttrain)
    pickle.dump(dict(split=sp, preds=preds), open(f"res_{which}.pkl", "wb"))
    print("done", which, time.time() - t0)


In [ ]:
%%writefile crc.py
"""Conformal risk control (CRC) for time-adaptive transient stability decisions."""
import numpy as np
from data import WINDOWS
TGRID = np.round(np.concatenate([np.arange(0.0, 12.0, 0.02), np.arange(12.0, 400.0001, 0.25)]), 4)

def decide(Mk, S, th_s, th_u):
    """Mk: (n,K) margins; S: (K,) scales. Returns decision (1 unstable / 0 stable), step index, undecided flag."""
    n, K = Mk.shape
    dec = np.ones(n, int); step = np.full(n, K - 1); und = np.ones(n, bool)
    for k in range(K):
        open_ = und.copy()
        st = open_ & (Mk[:, k] - th_s * S[k] > 0)
        un = open_ & ~st & (Mk[:, k] + th_u * S[k] < 0)
        dec[st] = 0; dec[un] = 1; step[st | un] = k; und[st | un] = False
    return dec, step, und

def crc_threshold(losses_fn, n, alpha, B=1.0):
    """smallest theta on grid with (n/(n+1)) Rhat + B/(n+1) <= alpha; losses non-increasing in theta."""
    for th in TGRID:
        R = losses_fn(th).mean()
        if n / (n + 1) * R + B / (n + 1) <= alpha: return th
    return TGRID[-1]

def calibrate(Mk, S, y, alpha, beta, rs=None):
    """Class-conditional (Mondrian) CRC. theta_u from the stable calibration cases (early-false-alarm bound <= beta);
    theta_s from the unstable calibration cases (false-dismissal rate <= alpha, conditional on instability)."""
    st, un = ~y, y
    def fa(th):
        return (((Mk[st] + th * S) < 0).any(1)).astype(float)
    th_u = crc_threshold(fa, st.sum(), beta)
    def fd(th):
        dec, _, _ = decide(Mk[un], S, th, th_u)
        return (dec == 0).astype(float)
    th_s = crc_threshold(fd, un.sum(), alpha)
    return th_s, th_u

def evaluate(Mk, S, y, th_s, th_u):
    dec, step, und = decide(Mk, S, th_s, th_u)
    t_ms = np.array(WINDOWS)[step] * 10.0
    return dict(fdr=float(((dec == 0) & y).sum() / max(y.sum(), 1)),
                far=float(((dec == 1) & ~y).sum() / max((~y).sum(), 1)),
                t_mean=float(t_ms.mean()), und=float(und.mean()),
                early=float((step == 0).mean()), acc=float((dec == y).mean()))


In [ ]:
%%writefile analyze.py
import numpy as np, pickle, json
from data import load_all, build, WINDOWS
from crc import calibrate, evaluate, decide
import exp
R = exp.R
DS_full = {W: build(R, W) for W in WINDOWS}
ID = DS_full[WINDOWS[-1]]["id"]
def _sub(D, keep):
    m = np.isin(D["id"], keep)
    return {k: (v[m] if isinstance(v, np.ndarray) and len(v) == len(D["id"]) else v) for k, v in D.items()}
DS = {W: _sub(DS_full[W], ID) for W in WINDOWS}
for W in WINDOWS: assert (DS[W]["id"] == ID).all()
def align(pred, key="m"):
    v = pred[key]; idx = {i: j for j, i in enumerate(pred["id"])}
    return np.array([v[idx[i]] for i in ID])
y_all = DS[WINDOWS[0]]["unstable"]; tsi_all = DS[WINDOWS[0]]["tsi"]
line_all = DS[WINDOWS[0]]["line"]
OUT = {}

def margins(res, kind, seed=0):
    Mk = np.stack([align(res["preds"][(kind, W, seed)]) for W in WINDOWS], 1)
    S = np.array([res["preds"][(kind, W, seed)]["scale"] for W in WINDOWS])
    return Mk, S

def gb_margins(res, name="GB"):
    Mk = np.stack([0.5 - align(res["preds"][(name, W, 0)], "p") for W in WINDOWS], 1)
    return Mk

def sel(ids_): return np.isin(ID, ids_)

def fixed_metrics(m, mask):
    y = y_all[mask]; yh = m[mask] < 0
    return dict(acc=(yh == y).mean(), fdr=(~yh & y).sum() / max(y.sum(), 1), far=(yh & ~y).sum() / max((~y).sum(), 1),
                mae=np.abs(m[mask] - tsi_all[mask]).mean())

def ta_consistency(res, mask):
    pu = np.stack([align(res["preds"][("GBu", W, 0)], "p") for W in WINDOWS], 1)[mask]
    ps = np.stack([align(res["preds"][("GBs", W, 0)], "p") for W in WINDOWS], 1)[mask]
    p = np.stack([align(res["preds"][("GB", W, 0)], "p") for W in WINDOWS], 1)[mask]
    n = mask.sum(); dec = np.full(n, -1); step = np.full(n, len(WINDOWS) - 1)
    for k in range(len(WINDOWS)):
        o = dec < 0
        a_u = o & (pu[:, k] > .5) & (ps[:, k] > .5); a_s = o & (pu[:, k] <= .5) & (ps[:, k] <= .5)
        dec[a_u] = 1; dec[a_s] = 0; step[a_u | a_s] = k
    last = dec < 0; dec[last] = (p[last, -1] > .5).astype(int)
    y = y_all[mask]
    return dict(fdr=((dec == 0) & y).sum() / max(y.sum(), 1), far=((dec == 1) & ~y).sum() / max((~y).sum(), 1),
                t_mean=(np.array(WINDOWS)[step] * 10.0).mean(), acc=(dec == y).mean())

def resplit_eval(Mk, S, cal_pool_fn, alpha, beta=0.05, n_rep=200, seed=0):
    rs = np.random.RandomState(seed); outs = []
    for r in range(n_rep):
        cal, test = cal_pool_fn(rs)
        ths, thu = calibrate(Mk[cal], S, y_all[cal], alpha, beta, rs)
        outs.append(evaluate(Mk[test], S, y_all[test], ths, thu))
    keys = outs[0].keys()
    return {k: (float(np.mean([o[k] for o in outs])), float(np.std([o[k] for o in outs]))) for k in keys} | \
           dict(viol=float(np.mean([o["fdr"] > alpha for o in outs])), fdr_all=[o["fdr"] for o in outs])


In [ ]:
%%writefile make_results.py
import numpy as np, pickle, json, os, sys
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from analyze import *
MM = 1 / 25.4
plt.rcParams.update({"font.family": ["Liberation Sans", "Arial", "DejaVu Sans"], "font.size": 8, "axes.linewidth": 0.6, "lines.linewidth": 1.1,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7, "legend.frameon": False,
    "mathtext.fontset": "custom", "mathtext.rm": "Liberation Sans", "mathtext.it": "Liberation Sans:italic"})
def lab(ax, s): ax.text(-0.2, 1.04, s, transform=ax.transAxes, fontsize=9, weight="bold")
def save(fig, n): fig.savefig(f"figs/{n}.png", dpi=600, bbox_inches="tight", facecolor="white"); fig.savefig(f"figs/{n}.pdf", bbox_inches="tight"); plt.close(fig)
STY = {"GB": dict(color="0.55", ls=":", marker="s"), "DD": dict(color="0.4", ls="--", marker="^"),
       "MT": dict(color="0.25", ls="-.", marker="o"), "PI": dict(color="black", ls="-", marker="D")}
res = {k: pickle.load(open(f"res_{k}.pkl", "rb")) for k in ("S1", "S2", "S3", "S2G", "S2U", "S2L", "S2F") if os.path.exists(f"res_{k}.pkl")}
O = {}
# ---------------- dataset statistics ----------------
taus = np.array([r["sc"]["tau"] for r in R]); lams = np.array([r["sc"]["lam"] for r in R])
rid = np.array([r["sc"]["id"] for r in R]); order = np.argsort(rid)
O["n"] = int(len(ID)); O["unstable_frac"] = float(y_all.mean()); O["n_sims_ok"] = len(R)
fig, axs = plt.subplots(1, 2, figsize=(174 * MM, 55 * MM))
d0 = DS[WINDOWS[0]]
ax = axs[0]; st = ~y_all
ax.scatter(d0["tau"][st] * 1000, d0["tsi"][st], s=5, facecolors="none", edgecolors="0.45", lw=0.5, label="stable")
ax.scatter(d0["tau"][~st] * 1000, d0["tsi"][~st], s=6, marker="x", color="black", lw=0.6, label="unstable")
ax.axhline(0, color="black", lw=0.6, ls=":"); ax.set_xlabel("Fault duration $\\tau$ / ms"); ax.set_ylabel("Transient stability index $\\eta$"); ax.legend(loc="center left", bbox_to_anchor=(0.0, 0.4)); lab(ax, "a")
ax = axs[1]
bins = np.linspace(0.8, 1.2, 9); c = 0.5 * (bins[1:] + bins[:-1])
fr = [y_all[(d0["lam"] >= a) & (d0["lam"] < b)].mean() for a, b in zip(bins[:-1], bins[1:])]
ax.bar(c, fr, width=0.045, color="0.8", edgecolor="black", lw=0.6)
for x0 in (1.04, 1.10): ax.axvline(x0, color="black", ls="--", lw=0.7)
ax.text(0.92, 0.9 * max(fr), "S3 train", ha="center", fontsize=7); ax.text(1.07, 0.9 * max(fr), "cal.", ha="center", fontsize=7); ax.text(1.15, 0.9 * max(fr), "test", ha="center", fontsize=7)
ax.set_xlabel("Loading factor $\\lambda$"); ax.set_ylabel("Share of unstable contingencies"); lab(ax, "b")
fig.tight_layout(w_pad=2); save(fig, "Fig2")
# ---------------- physics mismatch of true trajectories ----------------
from model import SwingNet
D10 = DS[10]; trn = sel(res["S2"]["split"]["train"])
Y, pm0, mask = D10["Y"][trn], D10["pm0"][trn], D10["mask"][trn]
m2 = (mask[:, 1:] * mask[:, :-1]).astype(bool)
rows = []
for name, K in (("swing only (P_m = P_e^0)", None), ("swing + full static droop", exp.Sn / (0.05 * 100)), ("swing + governor lead fraction (GRACE)", exp.K_GOV)):
    net = SwingNet(10, exp.H, exp.M, K)
    rA, rB = net.residual(Y, pm0)
    rows.append((name, float(np.sqrt((rB[m2] ** 2).mean())), float(np.sqrt((rB[m2] ** 2).mean(0)).min()), float(np.sqrt((rB[m2] ** 2).mean(0)).max())))
dP = Y[..., 2][:, 1:][m2]
O["phys_table"] = rows; O["dP_rms"] = float(np.sqrt((dP ** 2).mean())); O["rA_rms"] = float(np.sqrt((rA[m2] ** 2).mean()))
# ---------------- fixed-window metrics ----------------
def fixed_table(rk, test_mask, kinds, seeds):
    T = {}
    for kind in kinds:
        for W in WINDOWS:
            vals = []
            for s in seeds:
                key = (kind, W, s)
                if kind == "GB": key = ("GB", W, 0); m = 0.5 - align(rk["preds"][key], "p")
                else:
                    if key not in rk["preds"]: continue
                    m = align(rk["preds"][key])
                vals.append(fixed_metrics(m, test_mask))
                if kind == "GB": break
            if vals: T[(kind, W)] = {k: (float(np.mean([v[k] for v in vals])), float(np.std([v[k] for v in vals]))) for k in vals[0]}
    return T
masks = {}
if "S1" in res: masks["S1"] = sel(res["S1"]["split"]["test"])
masks["S2"] = sel(res["S2"]["split"]["heldout"])
if "S3" in res: masks["S3"] = sel(res["S3"]["split"]["test"])
O["fixed"] = {}
for sname, rk in res.items():
    if sname not in masks: continue
    seeds = [0, 1, 2] if sname == "S2" else [0]
    O["fixed"][sname] = {f"{k[0]}|{k[1]}": v for k, v in fixed_table(rk, masks[sname], ["GB", "DD", "MT", "PI"], seeds).items()}
if "S2G" in res:
    O["fixed"]["S2G"] = {f"{k[0]}|{k[1]}": v for k, v in fixed_table(res["S2G"], masks["S2"], ["PI0"], [0]).items()}
# Fig 3: S2 accuracy, FDR, MAE vs window
fig, axs = plt.subplots(1, 3, figsize=(174 * MM, 52 * MM)); F = O["fixed"]["S2"]
for kind, lbl in (("GB", "GB"), ("DD", "DD"), ("MT", "MT"), ("PI", "PI")):
    for j, met in enumerate(("acc", "fdr", "mae")):
        if kind == "GB" and met == "mae": continue
        mu = [F[f"{kind}|{W}"][met][0] for W in WINDOWS]; sd = [F[f"{kind}|{W}"][met][1] for W in WINDOWS]
        axs[j].errorbar(np.array(WINDOWS) * 10, mu, yerr=sd, ms=3.2, mfc="white", capsize=2, label=lbl, **STY[kind])
for ax, yl, l in zip(axs, ("Accuracy", "False-dismissal rate", "MAE of $\\hat{\\eta}$"), "abc"):
    ax.set_xlabel("Time after clearing / ms"); ax.set_ylabel(yl); lab(ax, l); ax.set_xticks([30, 60, 100, 150])
axs[0].legend(loc="center left", bbox_to_anchor=(0.0, 0.45), ncol=2); fig.tight_layout(w_pad=1.5); save(fig, "Fig3")
# ---------------- noise robustness (S2, seed 0) ----------------
O["noise"] = {}
for kind in ("DD", "MT", "PI"):
    for W in WINDOWS:
        p = res["S2"]["preds"][(kind, W, 0)]
        if p.get("m_noise") is not None:
            O["noise"][f"{kind}|{W}"] = dict(clean=fixed_metrics(align(p), masks["S2"]), noisy=fixed_metrics(align(p, "m_noise"), masks["S2"]))
# forecast physics consistency
O["forecast_phys"] = {k: res["S2"]["preds"][(k, 10, 0)]["phys"] for k in ("MT", "PI")}
O["true_res"] = res["S2"]["preds"][("PI", 10, 0)]["true_res"]
O["timing"] = {k: (res["S2"]["preds"][(k, 10, 0)]["ttrain"], res["S2"]["preds"][(k, 10, 0)]["tinf"]) for k in ("DD", "MT", "PI")}
O["timing"]["GB"] = (res["S2"]["preds"][("GB", 10, 0)]["ttrain"], None)
pickle.dump(O, open("results_partA.pkl", "wb"))
print(json.dumps({k: v for k, v in O.items() if k in ("n", "unstable_frac", "phys_table", "dP_rms", "rA_rms", "forecast_phys", "true_res", "timing")}, indent=1, default=str))


In [ ]:
%%writefile make_results_b.py
import numpy as np, pickle, json, os
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from analyze import *
from make_results import MM, lab, save, STY, res, masks
O = pickle.load(open("results_partA.pkl", "rb"))
sp2 = res["S2"]["split"]; HL = np.array(sp2["heldout_lines"])
heldout = sel(sp2["heldout"]); naive2 = np.where(sel(sp2["cal_naive"]))[0]
def s2_disjoint(rs):
    cl = rs.choice(HL, 7, replace=False)
    cal = np.where(heldout & np.isin(line_all, cl))[0]; test = np.where(heldout & ~np.isin(line_all, cl))[0]
    return cal, test
def s2_naive(rs):
    cl = rs.choice(HL, 7, replace=False)
    return naive2, np.where(heldout & ~np.isin(line_all, cl))[0]
def last_only(Mk, S): return Mk[:, -1:], S[-1:]
TA = {}
cands = {"PI": margins(res["S2"], "PI", 0), "DD": margins(res["S2"], "DD", 0)}
gbM = gb_margins(res["S2"]); gtr = sel(sp2["train"])
cands["GB"] = (gbM, np.array([max(np.median(np.abs(gbM[gtr, k] - (0.5 - y_all[gtr]))), 0.01) for k in range(len(WINDOWS))]))
for a in (0.05, 0.10, 0.15, 0.20):
    for name, (Mk, S) in cands.items():
        TA[("S2", "disjoint", name, a)] = resplit_eval(Mk, S, s2_disjoint, a)
        if a == 0.05: TA[("S2", "naive", name, a)] = resplit_eval(Mk, S, s2_naive, a)
    Mk, S = cands["DD"]; Ml, Sl = last_only(Mk, S)
    TA[("S2", "disjoint", "DD-150", a)] = resplit_eval(Ml, Sl, s2_disjoint, a)
    TA[("S2", "disjoint", "DD-150", a)]["t_mean"] = (150.0, 0.0)
# seeds 1,2 for GRACE at alpha 0.05
for s in (1, 2):
    for kd in ("DD", "PI"):
        Mk, S = margins(res["S2"], kd, s); TA[("S2", "disjoint", f"{kd}-s{s}", 0.05)] = resplit_eval(Mk, S, s2_disjoint, 0.05)
# TA-Cons on the same test-line resplits
rs = np.random.RandomState(0); tac = []
for r in range(200):
    _, test = s2_disjoint(rs); m = np.zeros(len(ID), bool); m[test] = True; tac.append(ta_consistency(res["S2"], m))
TA[("S2", "TA-Cons")] = {k: (float(np.mean([t[k] for t in tac])), float(np.std([t[k] for t in tac]))) for k in tac[0]}
# S1 random
if "S1" in res:
    sp1 = res["S1"]["split"]; pool = np.where(sel(np.concatenate([sp1["cal"], sp1["test"]])))[0]
    def s1_fn(rs):
        p = rs.permutation(pool); h = len(p) // 2; return p[:h], p[h:]
    for kd in ("DD", "PI"):
        Mk, S = margins(res["S1"], kd, 0); TA[("S1", "random", kd, 0.05)] = resplit_eval(Mk, S, s1_fn, 0.05)
    rs = np.random.RandomState(0); tac = []
    for r in range(200):
        _, test = s1_fn(rs); m = np.zeros(len(ID), bool); m[test] = True; tac.append(ta_consistency(res["S1"], m))
    TA[("S1", "TA-Cons")] = {k: (float(np.mean([t[k] for t in tac])), float(np.std([t[k] for t in tac]))) for k in tac[0]}
# S3 operating-point shift
if "S3" in res:
    sp3 = res["S3"]["split"]; c3 = np.where(sel(sp3["cal"]))[0]; t3 = np.where(sel(sp3["test"]))[0]; n3 = np.where(sel(sp3["cal_naive"]))[0]
    def s3_shift(rs): return rs.choice(c3, int(.8 * len(c3)), replace=False), rs.choice(t3, int(.8 * len(t3)), replace=False)
    def s3_naive(rs): return rs.choice(n3, int(.8 * len(n3)), replace=False), rs.choice(t3, int(.8 * len(t3)), replace=False)
    for kd in ("DD", "PI"):
        Mk, S = margins(res["S3"], kd, 0)
        TA[("S3", "shift-aware", kd, 0.05)] = resplit_eval(Mk, S, s3_shift, 0.05)
        TA[("S3", "naive", kd, 0.05)] = resplit_eval(Mk, S, s3_naive, 0.05)
    rs = np.random.RandomState(0); tac = []
    for r in range(200):
        _, test = s3_shift(rs); m = np.zeros(len(ID), bool); m[test] = True; tac.append(ta_consistency(res["S3"], m))
    TA[("S3", "TA-Cons")] = {k: (float(np.mean([t[k] for t in tac])), float(np.std([t[k] for t in tac]))) for k in tac[0]}
O["TA"] = TA
# ---------- Fig 4: FDR distributions ----------
fig, axs = plt.subplots(1, 2, figsize=(174 * MM, 56 * MM))
ax = axs[0]; data, labels = [], []
for name, lbl in (("DD", "DD net"), ("PI", "PI net"), ("GB", "GB")):
    for mode, ml in (("naive", "naive"), ("disjoint", "loc.-disj.")):
        data.append(TA[("S2", mode, name, 0.05)]["fdr_all"]); labels.append(f"{lbl}\n{ml}")
bp = ax.boxplot(data, widths=0.55, patch_artist=True, medianprops=dict(color="black"), flierprops=dict(marker=".", ms=2))
for j, b in enumerate(bp["boxes"]): b.set_facecolor("0.85" if j % 2 == 0 else "white")
ax.axhline(0.05, color="black", ls="--", lw=0.8); ax.text(6.45, 0.053, "$\\alpha$", fontsize=8)
ax.set_xticks(range(1, 7)); ax.set_xticklabels(labels, fontsize=6.5); ax.set_ylabel("Test false-dismissal rate"); lab(ax, "a")
ax = axs[1]; al = [0.05, 0.10, 0.15, 0.20]
for name, lbl, st in (("DD", "DD net + CRC", STY["PI"]), ("PI", "PI net + CRC", STY["DD"]), ("GB", "GB + CRC", STY["GB"]), ("DD-150", "DD net, 150 ms only", dict(color="0.5", ls="-", marker="v"))):
    ax.plot([TA[("S2", "disjoint", name, a)]["fdr"][0] for a in al], [TA[("S2", "disjoint", name, a)]["far"][0] for a in al], ms=3.5, mfc="white", label=lbl, **st)
tc = TA[("S2", "TA-Cons")]; ax.plot(tc["fdr"][0], tc["far"][0], "*", ms=8, color="black", label="TA-Cons [12]")
ax.plot([0, 0.2], [0, 0], alpha=0); ax.set_xlabel("Mean test false-dismissal rate"); ax.set_ylabel("Mean test false-alarm rate"); ax.legend(loc="upper right"); lab(ax, "b")
fig.tight_layout(w_pad=2); save(fig, "Fig4")
# ---------- Fig 5: decision time ----------
fig, axs = plt.subplots(1, 2, figsize=(174 * MM, 52 * MM))
ax = axs[0]
for name, lbl, st in (("DD", "DD net + CRC", STY["PI"]), ("PI", "PI net + CRC", STY["DD"]), ("GB", "GB + CRC", STY["GB"])):
    ax.plot(al, [TA[("S2", "disjoint", name, a)]["t_mean"][0] for a in al], ms=3.5, mfc="white", label=lbl, **st)
ax.axhline(tc["t_mean"][0], color="black", ls=":", lw=0.8); ax.text(0.2, tc["t_mean"][0] + 2, "TA-Cons", ha="right", fontsize=7)
ax.set_xlabel("Target $\\alpha$"); ax.set_ylabel("Mean decision time / ms"); ax.legend(loc="upper right"); lab(ax, "a")
ax = axs[1]
for name, lbl, st in (("DD", "DD net + CRC", STY["PI"]), ("PI", "PI net + CRC", STY["DD"]), ("GB", "GB + CRC", STY["GB"])):
    ax.plot(al, [100 * TA[("S2", "disjoint", name, a)]["und"][0] for a in al], ms=3.5, mfc="white", label=lbl, **st)
ax.set_xlabel("Target $\\alpha$"); ax.set_ylabel("Undecided at 150 ms (alarmed) / %"); lab(ax, "b")
fig.tight_layout(w_pad=2); save(fig, "Fig5")
# ---------- Fig 6: forecast examples ----------
p_pi = align(res["S2"]["preds"][("PI", 10, 0)], "traj"); p_mt = align(res["S2"]["preds"][("MT", 10, 0)], "traj")
D10 = DS[10]; ho = np.where(heldout)[0]
cand_s = ho[(~y_all[ho]) & (D10["tsi"][ho] < 0.35)]; cand_u = ho[y_all[ho]]
fig, axs = plt.subplots(1, 2, figsize=(174 * MM, 55 * MM))
for ax, cs, l, ttl in ((axs[0], cand_s, "a", "stable"), (axs[1], cand_u, "b", "unstable")):
    j = cs[0] if len(cs) else ho[0]
    g = int(np.argmax(np.abs(D10["Y"][j, :, :, 0]).max(0)[:9]))
    t_win = (np.arange(10) - 9) * 10; t_f = np.arange(exp.H + 1) * 10
    ax.plot(t_win, np.degrees(D10["X"][j, :, g, 0]), color="black", lw=1.6, label="measured window")
    msk = D10["mask"][j] > 0
    ax.plot(t_f[msk], np.degrees(D10["Y"][j, msk, g, 0]), color="black", lw=1.0, label="true future")
    ax.plot(t_f, np.degrees(p_pi[j, :, g, 0]), color="0.3", ls="--", label="PI forecast")
    ax.plot(t_f, np.degrees(p_mt[j, :, g, 0]), color="0.6", ls=":", lw=1.4, label="MT forecast")
    ax.set_xlabel("Time from decision instant ($t_c$ + 100 ms) / ms"); ax.set_ylabel(f"COI angle of G{g+1} / deg")
    ax.text(0.03, 0.92, f"{ttl}: $\\eta$ = {D10['tsi'][j]:.2f}, $\\hat{{\\eta}}$ = {align(res['S2']['preds'][('PI', 10, 0)])[j]:.2f}", transform=ax.transAxes, fontsize=7)
    lab(ax, l)
axs[0].legend(loc="lower right"); fig.tight_layout(w_pad=2); save(fig, "Fig6")
for k in ("S2U", "S2L", "S2F"):
    if k in res: O[k] = {str(kk): (fixed_metrics(align(v), masks["S2"]) if "m" in v else None) for kk, v in res[k]["preds"].items() if isinstance(v, dict) and "m" in v}
if "S2U" in res:
    pool = res["S2U"]["preds"][("pool_ids",)]; tmask = heldout & ~np.isin(ID, pool)
    O["S2U_test"] = {"PIU|" + str(W): fixed_metrics(align(res["S2U"]["preds"][("PIU", W, 0)]), tmask) for W in WINDOWS}
    O["S2U_ref"] = {"PI|" + str(W): fixed_metrics(align(res["S2"]["preds"][("PI", W, 0)]), tmask) for W in WINDOWS}
pickle.dump(O, open("results_all.pkl", "wb"))
print("saved")


In [ ]:
%%writefile fig1.py
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
plt.rcParams.update({"font.family": ["Liberation Sans", "Arial", "DejaVu Sans"], "font.size": 8, "mathtext.fontset": "custom",
    "mathtext.rm": "Liberation Sans", "mathtext.it": "Liberation Sans:italic"})
fig, ax = plt.subplots(figsize=(7.1, 3.6)); ax.set_xlim(0, 20); ax.set_ylim(0, 10); ax.axis("off")
def box(x, y, w, h, t, fc="white", ls="-", fs=7.2, bold=False):
    ax.add_patch(FancyBboxPatch((x - w/2, y - h/2), w, h, boxstyle="round,pad=0.02,rounding_size=0.15", fc=fc, ec="black", lw=0.8, ls=ls))
    ax.text(x, y, t, ha="center", va="center", fontsize=fs, linespacing=1.35, weight="bold" if bold else "normal")
def arr(p, q, ls="-"): ax.add_patch(FancyArrowPatch(p, q, arrowstyle="-|>", mutation_scale=8, lw=0.8, color="black", ls=ls, shrinkA=0, shrinkB=0))
ax.text(0.2, 9.55, "(a) Offline training", fontsize=8.5, weight="bold")
ax.text(0.2, 4.35, "(b) Online time-adaptive decision", fontsize=8.5, weight="bold")
box(2.05, 7.4, 3.9, 2.5, "PMU data after clearing\n$\\tilde{\\delta}_i$, $\\Delta\\omega_i$, $\\Delta P_{e,i}$\n(COI frame, $W$ samples)\n+ pre-fault $P_{e,i}^{0}$", fs=6.8)
box(6.6, 7.4, 3.2, 2.4, "Shared trunk\n(MLP)", fc="0.93", bold=True)
arr((4.0, 7.4), (5.0, 7.4))
box(10.9, 8.55, 3.9, 1.2, "Margin head: $\\hat{\\eta}$ (TSI)")
box(10.9, 6.25, 3.9, 1.2, "Forecast head:\n$\\hat{\\delta},\\hat{\\omega},\\Delta\\hat{P}_e$ over 0.25 s")
arr((8.2, 8.0), (8.95, 8.45)); arr((8.2, 6.8), (8.95, 6.35))
box(16.6, 8.55, 6.2, 1.2, "$L_{\\eta}$: Huber loss on simulated TSI")
box(16.6, 6.25, 6.2, 2.2, "$L_{\\mathrm{phys}}$: governor-aware swing residual\n$M_i\\,\\dot{\\omega}_i + \\Delta P_{e,i} + K_i\\,\\Delta\\omega_i = 0$\n$\\dot{\\tilde{\\delta}}_i - \\omega_b(\\Delta\\omega_i - \\Delta\\omega_{\\mathrm{COI}}) = 0$\n+ $L_{\\mathrm{traj}}$ (forecast error)", fs=6.9)
arr((12.85, 8.55), (13.5, 8.55)); arr((12.85, 6.25), (13.5, 6.25))
ax.plot([0.2, 19.8], [4.75, 4.75], color="0.5", lw=0.6, ls=":")
xs = [2.0, 5.6, 9.2, 12.8]; lab = ["30 ms", "60 ms", "100 ms", "150 ms"]
for k, (x, l) in enumerate(zip(xs, lab)):
    box(x, 3.1, 3.0, 1.35, f"$t_c$ + {l}\n$\\hat{{\\eta}}_{k+1}$, scale $s_{k+1}$", fs=7)
    if k < 3: arr((x + 1.5, 3.1), (xs[k+1] - 1.5, 3.1))
    ax.text(x, 1.85, "wait" if k < 3 else "", ha="center", fontsize=6.5, style="italic", color="0.35")
box(17.2, 3.55, 4.9, 1.05, "Stable if $\\hat{\\eta}_k - \\hat{\\theta}_s s_k > 0$", fs=7)
box(17.2, 2.2, 4.9, 1.05, "Unstable if $\\hat{\\eta}_k + \\hat{\\theta}_u s_k < 0$", fs=7)
box(17.2, 0.85, 5.1, 1.0, "Undecided at 150 ms: alarm", fs=7, ls="--")
arr((14.3, 3.2), (14.75, 3.5)); arr((14.3, 3.0), (14.75, 2.3))
ax.text(17.2, 4.35, "rule checked at every step $k$", ha="center", fontsize=6.5, style="italic", color="0.35")
ax.text(8.6, 0.55, "$\\hat{\\theta}_s,\\hat{\\theta}_u$ from conformal risk control on location-disjoint calibration data:\n$\\mathbb{E}[\\mathrm{false\\ dismissal}] \\leq \\alpha$ (finite-sample)", ha="center", fontsize=7, style="italic")
fig.savefig("figs/Fig1.png", dpi=600, bbox_inches="tight", facecolor="white"); fig.savefig("figs/Fig1.pdf", bbox_inches="tight")


In [ ]:
%%writefile fig_ta.py
import pickle, numpy as np, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
MM = 1/25.4
plt.rcParams.update({"font.family": ["Liberation Sans", "Arial", "DejaVu Sans"], "font.size": 8, "axes.linewidth": 0.6, "lines.linewidth": 1.1,
    "xtick.labelsize": 7.5, "ytick.labelsize": 7.5, "legend.fontsize": 7, "legend.frameon": False,
    "mathtext.fontset": "custom", "mathtext.rm": "Liberation Sans", "mathtext.it": "Liberation Sans:italic"})
STY = {"GB": dict(color="0.55", ls=":", marker="s"), "DD": dict(color="0.4", ls="--", marker="^"), "PI": dict(color="black", ls="-", marker="D")}
TA = pickle.load(open("results_all.pkl", "rb"))["TA"]
def lab(ax, s): ax.text(-0.2, 1.04, s, transform=ax.transAxes, fontsize=9, weight="bold")
fig, axs = plt.subplots(1, 2, figsize=(174 * MM, 56 * MM))
ax = axs[0]; data, labels = [], []
for name, lbl in (("DD", "DD net"), ("PI", "PI net"), ("GB", "GB")):
    for mode, ml in (("naive", "naive"), ("disjoint", "loc.-disj.")):
        data.append(TA[("S2", mode, name, 0.05)]["fdr_all"]); labels.append(f"{lbl}\n{ml}")
bp = ax.boxplot(data, widths=0.55, patch_artist=True, medianprops=dict(color="black"), flierprops=dict(marker=".", ms=2))
for j, b in enumerate(bp["boxes"]): b.set_facecolor("0.85" if j % 2 == 0 else "white")
ax.axhline(0.05, color="black", ls="--", lw=0.8); ax.text(6.45, 0.06, "$\\alpha$", fontsize=8)
ax.set_xticks(range(1, 7)); ax.set_xticklabels(labels, fontsize=6.5); ax.set_ylabel("Test false-dismissal rate"); lab(ax, "a")
ax = axs[1]; al = [0.05, 0.10, 0.15, 0.20]
for name, lbl, st in (("DD", "GRACE (DD)", STY["PI"]), ("PI", "GRACE (PI)", STY["DD"]), ("GB", "GRACE (GB)", STY["GB"]), ("DD-150", "GRACE (DD), 150 ms only", dict(color="0.5", ls="-", marker="v"))):
    ax.plot([TA[("S2", "disjoint", name, a)]["fdr"][0] for a in al], [TA[("S2", "disjoint", name, a)]["far"][0] for a in al], ms=3.5, mfc="white", label=lbl, **st)
tc = TA[("S2", "TA-Cons")]; ax.plot(tc["fdr"][0], tc["far"][0], "*", ms=8, color="black", label="TA-Cons")
ax.set_xlim(0, 0.23); ax.set_xlabel("Mean test false-dismissal rate"); ax.set_ylabel("Mean test false-alarm rate"); ax.legend(loc="upper right"); lab(ax, "b")
fig.tight_layout(w_pad=2); fig.savefig("paper/Fig5.png", dpi=600, bbox_inches="tight", facecolor="white"); fig.savefig("figs/Fig5_final.pdf", bbox_inches="tight")
# Fig 6 (decision time) with GRACE labels
fig, axs = plt.subplots(1, 2, figsize=(174 * MM, 52 * MM))
for j, (key, yl) in enumerate((("t_mean", "Mean decision time / ms"), ("und", "Undecided at 150 ms (alarmed) / %"))):
    ax = axs[j]
    for name, lbl, st in (("DD", "GRACE (DD)", STY["PI"]), ("PI", "GRACE (PI)", STY["DD"]), ("GB", "GRACE (GB)", STY["GB"])):
        f = 100 if key == "und" else 1
        ax.plot(al, [f * TA[("S2", "disjoint", name, a)][key][0] for a in al], ms=3.5, mfc="white", label=lbl, **st)
    ax.set_xlabel("Target $\\alpha$"); ax.set_ylabel(yl); lab(ax, "ab"[j])
axs[0].axhline(tc["t_mean"][0], color="black", ls=":", lw=0.8); axs[0].text(0.2, tc["t_mean"][0] + 2, "TA-Cons", ha="right", fontsize=7)
axs[0].legend(loc="upper right"); fig.tight_layout(w_pad=2); fig.savefig("paper/Fig6.png", dpi=600, bbox_inches="tight", facecolor="white")


## 5. Simulate the contingencies with ANDES

In [ ]:
#@title 5a. Export the IEEE 39-bus case once (faster loading)
import andes
andes.config_logger(stream_level=50)
if not os.path.exists('ieee39.json'):
    ss = andes.load(andes.get_case('ieee39/ieee39_full.xlsx'), setup=False, no_output=True, default_config=True)
    andes.io.dump(ss, 'json', 'ieee39.json', overwrite=True)
print("case ready")

In [ ]:
#@title 5b. Run the simulations (resumable – just rerun this cell after a disconnect)
!python gen.py {N_SIMS} {os.cpu_count()}
import glob; print(len(glob.glob('sims/*.pkl')), "contingencies on disk")

In [ ]:
#@title 5c. Quick data check
import numpy as np
from data import load_all
R = load_all()
print(len(R), "valid contingencies;", round(100 * np.mean([r['unstable'] for r in R]), 1), "% unstable")

## 6. Train predictors and baselines
S1 = random split, S2 = unseen fault lines (3 seeds), S3 = unseen loading, S2G = PI without governor term, S2U = label-free physics pool, S2L = physics-weight sensitivity, S2F = reduced training data.

In [ ]:
#@title 6. Training (about 10–20 min; finished experiments are skipped)
import subprocess
for mode in ["S2", "S1", "S3", "S2G", "S2U", "S2L", "S2F"]:
    if os.path.exists(f"res_{mode}.pkl"):
        print(mode, "already done"); continue
    print("running", mode, flush=True)
    r = subprocess.run(["python", "exp.py", mode, str(ITERS)], capture_output=True, text=True)
    print(r.stdout[-300:], r.stderr[-800:])

## 7. Analysis: fixed-instant metrics, conformal risk control (200 re-splits) and figures

In [ ]:
#@title 7a. Results part A (data set, physics residuals, fixed-instant metrics)
!python make_results.py 2>&1 | grep -v -i "warn\|findfont" | tail -5

In [ ]:
#@title 7b. Results part B (time-adaptive conformal risk control, about 5–10 min) and final figures
!python make_results_b.py 2>&1 | grep -v -i "warn\|findfont" | tail -3
!python fig1.py 2>/dev/null
!python fig_ta.py 2>/dev/null
import shutil   # figure numbering used in the manuscript
for src, dst in [("figs/Fig1.png", "paper/Fig1.png"), ("figs/Fig2.png", "paper/Fig2.png"),
                 ("figs/Fig3.png", "paper/Fig3.png"), ("figs/Fig6.png", "paper/Fig4.png")]:
    shutil.copy(src, dst)
print("figures saved in", os.path.join(WORKDIR, "paper"))

In [ ]:
#@title 8. Key numbers (compare with Tables 2–4 of the manuscript)
import pickle
O = pickle.load(open("results_all.pkl", "rb"))
print("Contingencies:", O["n"], "| unstable share:", round(O["unstable_frac"], 3))
print("\nTable 2 – swing residual of the true trajectories (RMS, p.u.):")
for name, rms, lo, hi in O["phys_table"]:
    print(f"  {name:42s} {rms:.3f}  ({lo:.2f}–{hi:.2f})")
print("\nTable 3 – S2 (unseen lines), accuracy / FDR at 150 ms:")
for k in ("GB", "DD", "MT", "PI"):
    v = O["fixed"]["S2"][f"{k}|15"]; print(f"  {k:3s} acc {v['acc'][0]:.3f}  FDR {v['fdr'][0]:.3f}")
print("\nTable 4 – time-adaptive decisions (alpha = 0.05):")
TA = O["TA"]
for key in [("S2", "disjoint", "DD", 0.05), ("S2", "naive", "DD", 0.05), ("S2", "disjoint", "DD-150", 0.05),
            ("S2", "disjoint", "PI", 0.05), ("S2", "disjoint", "GB", 0.05), ("S1", "random", "DD", 0.05),
            ("S3", "shift-aware", "DD", 0.05)]:
    v = TA[key]
    print(f"  {str(key):42s} FDR {v['fdr'][0]:.3f}  FAR {v['far'][0]:.3f}  t {v['t_mean'][0]:5.1f} ms  share>alpha {v['viol']:.2f}")
for s in ("S1", "S2", "S3"):
    v = TA[(s, "TA-Cons")]; print(f"  TA-Cons {s}: FDR {v['fdr'][0]:.3f}  FAR {v['far'][0]:.3f}")

In [ ]:
#@title 9. Show the figures
from IPython.display import Image, display
for i in range(1, 7):
    print(f"Fig. {i}"); display(Image(f"paper/Fig{i}.png", width=900))

In [ ]:
#@title 10. Download everything as a zip
!zip -qr GRACE_results.zip paper figs res_*.pkl results_all.pkl *.py
from google.colab import files
files.download("GRACE_results.zip")